# III. Economic news

mels notes for later
economic 

In [109]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score, confusion_matrix
import numpy as np
from IPython.display import display

In [131]:
calendar_data = pd.read_csv("../data/raw/19-21-calendar.csv", delimiter=";", header=None, names=[
    'date', 'time', 'country', 'impact', 'event', 'forecast', 'unit', 'actual', 'previous', 'revised_from'
])

# Strip leading/trailing spaces from all columns
calendar_data = calendar_data.apply(lambda x: x.str.strip() if isinstance(x, str) else x)

# Combine date and time into a single datetime column
calendar_data['datetime'] = pd.to_datetime(calendar_data['date'] + ' ' + calendar_data['time'])

# Round down datetime to the nearest hour
calendar_data['datetime'] = calendar_data['datetime'].dt.floor('H')

# Filter relevant columns
calendar_data = calendar_data[['datetime', 'country', 'impact', 'event', 'actual', 'forecast', 'previous']]

# Filter relevant countries
countries_of_interest = ['USD', 'Germany', 'EUR', 'France', 'Spain', 'Italy', 'G20', 'G7', 'IMF']
calendar_data = calendar_data[calendar_data['country'].str.contains('|'.join(countries_of_interest))]

# Load EURUSD price data
price_data = pd.read_csv("../data/raw/EUR_USD_H1_combined.csv")

# Filter relevant columns
price_data = price_data[['time', 'close']]
price_data['time'] = pd.to_datetime(price_data['time'])

# Ensure both datetime columns are in UTC
calendar_data['datetime'] = calendar_data['datetime'].dt.tz_localize('UTC')
price_data['time'] = price_data['time'].dt.tz_convert('UTC')

# Filter data for the years 2019 to 2021
calendar_data = calendar_data[(calendar_data['datetime'] >= '2019-01-01') & (calendar_data['datetime'] <= '2021-12-31')]
price_data = price_data[(price_data['time'] >= '2019-01-01') & (price_data['time'] <= '2021-12-31')]

# Merge datasets using the floored datetime for economic data and exact time for price data
merged_data = pd.merge(price_data, calendar_data, left_on='time', right_on='datetime', how='left')
merged_data = merged_data.drop(columns=['datetime'])
merged_data = merged_data.rename(columns={'time':'datetime'}).set_index("datetime")
impact_mapping = {
    'Low Volatility Expected': 1, 
    'Moderate Volatility Expected': 2, 
    'High Volatility Expected': 3,
    'Unknown': 0,
    '': 0,  # Handling any empty strings
}

forecast_mapping = {
    "Worse Than Expected": -1,
    "In Line with Expectation": 0,
    "Better Than Expected": 1,
    "0": 0,
    "": 0,
}
merged_data['impact'] = merged_data['impact'].str.strip().map(impact_mapping).fillna(0).astype(int)
merged_data['forecast'] = merged_data['forecast'].str.strip().map(forecast_mapping).fillna(0).astype(int)
# merged_data['change_from_forecast'] = np.where(merged_data['actual'] > merged_data['forecast'], 1, -1)

# Create a binary feature for change from previous
merged_data['change'] = np.where(merged_data['actual'] > merged_data['previous'], 1, -1)

merged_data['actual'].fillna(0, inplace=True)
merged_data['forecast'].fillna(0, inplace=True)
merged_data['previous'].fillna(0, inplace=True)

merged_data['price_direction'] = np.where(merged_data['close'].diff() > 0, 1, -1)
merged_data['pips'] = merged_data['close'].diff() * 10000
merged_data.dropna(inplace=True)
display(merged_data.head(30))
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)
pd.set_option('display.width', 800)


,close,country,impact,event,actual,forecast,previous,change,price_direction,pips
datetime,,,,,,,,,,
2019-01-02 03:00:00+00:00,1.14456,Spain,2,Spanish Manufacturing PMI ...,51.100,-1,52.4,-1,-1,-0.5
2019-01-02 03:00:00+00:00,1.14456,Italy,2,Italian Manufacturing PMI ...,49.200,1,48.4,1,-1,0.0
2019-01-02 03:00:00+00:00,1.14456,France,2,French Manufacturing PMI ...,49.700,0,49.7,-1,-1,0.0
2019-01-02 03:00:00+00:00,1.14456,Germany,3,German Manufacturing PMI ...,51.500,0,51.5,-1,-1,0.0
2019-01-02 09:00:00+00:00,1.14338,France,1,French 12-Month BTF Auction ...,-0.566,0,0.0,-1,-1,-15.0
2019-01-02 09:00:00+00:00,1.14338,France,1,French 3-Month BTF Auction ...,-0.570,0,0.0,-1,-1,0.0
2019-01-02 09:00:00+00:00,1.14338,France,1,French 6-Month BTF Auction ...,-0.569,0,0.0,-1,-1,0.0
2019-01-03 03:00:00+00:00,1.13646,Spain,2,Spanish Unemployment Change ...,-50.600,0,0.0,-1,1,5.9
2019-01-03 04:00:00+00:00,1.13670,Spain,1,Spanish Consumer Confidence ...,90.900,0,0.0,-1,1,2.4


Now we have the data we need to proceed. In the above dataframe, we've added a few columns and mappped the string impact labels to numeric values, as well as given postive or negative values to determine price direction After each news event. 

In [135]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

features = merged_data[['impact', 'actual', 'forecast', 'previous', 'change', 'price_direction', 'pips']]
label_classification = merged_data['price_direction']
label_regression = merged_data['pips']

X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(features, label_classification, test_size=0.2, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(features, label_regression, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_c = scaler.fit_transform(X_train_c)
X_test_c = scaler.fit_transform(X_test_c)
X_train_r = scaler.fit_transform(X_train_r)
X_test_r = scaler.fit_transform(X_test_r)


In [137]:
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor


classified = RandomForestClassifier(n_estimators=100, random_state=42)
classified.fit(X_train_c, y_train_c)
y_prediction_c = classified.predict(X_test_c)
display(y_prediction_c)


NameError: name 'classfieid' is not defined